In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from lifelines import CoxPHFitter
from IPython.display import HTML

np.random.seed(42)
n = 700

df = pd.DataFrame({
    "dist_km": np.random.gamma(shape=2.2, scale=120, size=n),
    "expressa": np.random.binomial(1, 0.35, size=n),
    "feriado": np.random.binomial(1, 0.18, size=n),
    "carrier_risco": np.random.binomial(1, 0.30, size=n),
    "despacho_fds": np.random.binomial(1, 0.22, size=n)
})

df["dist_100km"] = df["dist_km"] / 100.0

beta = {
    "dist_100km": -0.18,
    "expressa": 0.55,
    "feriado": -0.35,
    "carrier_risco": -0.45,
    "despacho_fds": -0.20
}

linpred = (
    beta["dist_100km"] * df["dist_100km"] +
    beta["expressa"] * df["expressa"] +
    beta["feriado"] * df["feriado"] +
    beta["carrier_risco"] * df["carrier_risco"] +
    beta["despacho_fds"] * df["despacho_fds"]
)

base_rate = 0.22
u = np.random.uniform(size=n)
event_time = -np.log(u) / (base_rate * np.exp(linpred))

censor_time = np.random.uniform(5.5, 10.0, size=n)

df["tempo"] = np.minimum(event_time, censor_time)
df["evento"] = (event_time <= censor_time).astype(int)

model_df = df[["tempo", "evento", "dist_100km", "expressa", "feriado", "carrier_risco", "despacho_fds"]].copy()

cph = CoxPHFitter()
cph.fit(model_df, duration_col="tempo", event_col="evento")

summary = cph.summary.copy()
summary["HR"] = np.exp(summary["coef"])
summary["HR_low"] = np.exp(summary["coef lower 95%"])
summary["HR_high"] = np.exp(summary["coef upper 95%"])

label_map = {
    "dist_100km": "Distância (+100 km)",
    "expressa": "Frete expresso",
    "feriado": "Despacho em feriado",
    "carrier_risco": "Transportadora risco",
    "despacho_fds": "Despacho em fim de semana"
}

order = ["expressa", "dist_100km", "carrier_risco", "feriado", "despacho_fds"]
summary = summary.loc[order].copy()
summary["label"] = [label_map[i] for i in summary.index]

cenario_padrao = pd.DataFrame([{
    "dist_100km": 3.5,
    "expressa": 0,
    "feriado": 0,
    "carrier_risco": 0,
    "despacho_fds": 0
}])

cenario_critico = pd.DataFrame([{
    "dist_100km": 5.5,
    "expressa": 0,
    "feriado": 1,
    "carrier_risco": 1,
    "despacho_fds": 1
}])

cenario_expressa = pd.DataFrame([{
    "dist_100km": 3.5,
    "expressa": 1,
    "feriado": 0,
    "carrier_risco": 0,
    "despacho_fds": 0
}])

surv_padrao = cph.predict_survival_function(cenario_padrao)
surv_critico = cph.predict_survival_function(cenario_critico)
surv_expressa = cph.predict_survival_function(cenario_expressa)

times = surv_padrao.index.values
vals_padrao = surv_padrao.iloc[:, 0].values
vals_critico = surv_critico.iloc[:, 0].values
vals_expressa = surv_expressa.iloc[:, 0].values

fig = plt.figure(figsize=(10.2, 5.8), facecolor="white")
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.35])

ax_hr = fig.add_subplot(gs[0, 0])
ax_surv = fig.add_subplot(gs[0, 1])

plt.subplots_adjust(left=0.22, right=0.97, top=0.84, bottom=0.14, wspace=0.30)

green_neon = "#39ff14"
red_neon = "#ff3131"
blue_neon = "#00e5ff"
purple_neon = "#b026ff"
burgundy = "#800020"
dark = "#1f1f1f"
gray = "#9e9e9e"

total_frames = 80

def draw_frame(frame):
    ax_hr.clear()
    ax_surv.clear()

    fig.suptitle(
        "Tempo até a entrega: abordagem com modelo de Cox",
        fontsize=16,
        fontweight="bold",
        y=0.95
    )

    n_vars = len(summary)
    shown_vars = min(n_vars, max(1, frame // 10 + 1))

    sub = summary.iloc[:shown_vars].copy()
    y_pos = np.arange(len(sub))[::-1]

    colors = [green_neon if hr > 1 else red_neon for hr in sub["HR"]]

    ax_hr.barh(y_pos, sub["HR"], color=colors, alpha=0.88)
    ax_hr.axvline(1.0, color=dark, linestyle="--", linewidth=1.2)

    for y, hr, lo, hi in zip(y_pos, sub["HR"], sub["HR_low"], sub["HR_high"]):
        ax_hr.plot([lo, hi], [y, y], color=dark, linewidth=1)
        ax_hr.plot([lo, lo], [y - 0.08, y + 0.08], color=dark, linewidth=1)
        ax_hr.plot([hi, hi], [y - 0.08, y + 0.08], color=dark, linewidth=1)

    ax_hr.set_yticks(y_pos)
    ax_hr.set_yticklabels(sub["label"], fontsize=8.8)
    ax_hr.set_xlim(0, max(2.0, summary["HR_high"].max() * 1.12))
    ax_hr.set_xlabel("Hazard ratio", fontsize=10)
    ax_hr.set_title("Efeito estimado dos fatores logísticos", fontsize=12, pad=10, fontweight="bold")
    ax_hr.grid(axis="x", alpha=0.18)

    ax_hr.text(
        0.02, 0.04,
        "HR > 1: entrega tende a acontecer antes\nHR < 1: entrega tende a demorar mais",
        transform=ax_hr.transAxes,
        fontsize=8.3,
        ha="left",
        va="bottom"
    )

    frac = min(1, (frame + 1) / 55)
    n_pts = max(2, int(len(times) * frac))

    ax_surv.plot(
        times[:n_pts], vals_padrao[:n_pts],
        color=blue_neon, linewidth=2.4, label="Cenário padrão"
    )
    ax_surv.plot(
        times[:n_pts], vals_expressa[:n_pts],
        color=purple_neon, linewidth=2.4, label="Cenário com expressa"
    )
    ax_surv.plot(
        times[:n_pts], vals_critico[:n_pts],
        color=burgundy, linewidth=2.4, label="Cenário crítico"
    )

    ax_surv.set_xlim(0, times.max())
    ax_surv.set_ylim(0, 1.02)
    ax_surv.set_xlabel("Dias", fontsize=10)
    ax_surv.set_ylabel("Probabilidade de ainda não ter sido entregue", fontsize=10)
    ax_surv.set_title("Curvas previstas para diferentes cenários", fontsize=12, pad=10, fontweight="bold")
    ax_surv.grid(alpha=0.18)
    ax_surv.legend(frameon=False, loc="upper right", fontsize=9)

    if frame >= 45:
        ax_surv.text(
            0.03, 0.10,
            "Cenários com risco operacional acumulado\n"
            "mantêm maior probabilidade de não entrega\n"
            "ao longo do tempo.",
            transform=ax_surv.transAxes,
            fontsize=8.8,
            ha="left",
            va="bottom"
        )

anim = FuncAnimation(fig, draw_frame, frames=total_frames, interval=220, repeat=True)

HTML(anim.to_jshtml())